# AeroPulse Model 4 — Pollution Propagation Forecast

**Purpose:** forecast the PM2.5 field for 0–3h nowcast and 6/12/24/48h horizons using the LLD's MVP hybrid design:

`current pollution field → spatial smoothing → wind-vector advection → atmospheric dilution/stability → CAMS correction → ML residual correction → forecast field`.

This notebook implements the baseline as a grid/time-series residual model. It is deliberately structured so ConvLSTM/TFT/GNN/physics-informed models can replace the residual learner later without changing the feature contract.

In [ ]:
!pip -q install pandas numpy scikit-learn lightgbm joblib requests pyarrow scipy

In [ ]:
# Core
import os, io, json, math, time, warnings, hashlib
from pathlib import Path
from datetime import datetime, timedelta, timezone
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path(os.getenv("AEROPULSE_DATA_DIR", "./data"))
ARTIFACT_DIR = Path(os.getenv("AEROPULSE_ARTIFACT_DIR", "./artifacts"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

REGION = {
    "min_lat": float(os.getenv("AEROPULSE_MIN_LAT", "28.0")),
    "max_lat": float(os.getenv("AEROPULSE_MAX_LAT", "32.8")),
    "min_lon": float(os.getenv("AEROPULSE_MIN_LON", "74.0")),
    "max_lon": float(os.getenv("AEROPULSE_MAX_LON", "78.5")),
}

OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY", "")
FIRMS_MAP_KEY = os.getenv("FIRMS_MAP_KEY", "")

def http_get_json(url, params=None, headers=None, timeout=60):
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def require_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value

print("AeroPulse ML environment initialized")
print("Region:", REGION)

In [ ]:
# OpenAQ v3 connector. OpenAQ exposes original measurements plus hourly aggregates.
# Use API key through OPENAQ_API_KEY.
OPENAQ_BASE = "https://api.openaq.org/v3"

def openaq_locations(bbox=REGION, limit=1000):
    params = {
        "bbox": f'{bbox["min_lon"]},{bbox["min_lat"]},{bbox["max_lon"]},{bbox["max_lat"]}',
        "limit": min(limit, 1000),
        "page": 1,
    }
    headers = {"X-API-Key": require_env("OPENAQ_API_KEY")}
    return http_get_json(f"{OPENAQ_BASE}/locations", params=params, headers=headers)

def openaq_location_hours(location_id, date_from, date_to, limit=1000):
    headers = {"X-API-Key": require_env("OPENAQ_API_KEY")}
    params = {
        "datetime_from": pd.Timestamp(date_from, tz="UTC").isoformat(),
        "datetime_to": pd.Timestamp(date_to, tz="UTC").isoformat(),
        "limit": limit,
        "page": 1,
    }
    return http_get_json(f"{OPENAQ_BASE}/locations/{int(location_id)}/hours",
                         params=params, headers=headers)

def discover_pm25_locations(max_locations=100):
    raw = openaq_locations(limit=max_locations)
    rows = raw.get("results", [])
    out = []
    for x in rows:
        params = x.get("parameters", [])
        names = {str(p.get("name", "")).lower() for p in params}
        if "pm25" in names or "pm2.5" in names:
            coords = x.get("coordinates") or {}
            if coords.get("latitude") is not None and coords.get("longitude") is not None:
                out.append({
                    "location_id": x["id"],
                    "name": x.get("name"),
                    "lat": coords["latitude"],
                    "lon": coords["longitude"],
                    "parameters": list(names),
                })
    return pd.DataFrame(out)

def normalize_openaq_hours(payload):
    rows = payload.get("results", [])
    out = []
    for r in rows:
        period = r.get("period", {})
        dt = (period.get("datetimeFrom") or {}).get("utc")
        param = r.get("parameter", {})
        out.append({
            "timestamp": pd.to_datetime(dt, utc=True, errors="coerce"),
            "parameter": str(param.get("name", "")).lower(),
            "value": safe_float(r.get("value")),
            "unit": r.get("unit"),
            "location_id": r.get("locationsId"),
        })
    return pd.DataFrame(out).dropna(subset=["timestamp"])

In [ ]:
# Open-Meteo is used here as a reproducible public weather/ERA5 access path.
# In production, replace this adapter with the IMD connector defined in the AeroPulse LLD.
OPEN_METEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
OPEN_METEO_FORECAST = "https://api.open-meteo.com/v1/forecast"

WEATHER_VARS = [
    "temperature_2m", "relative_humidity_2m", "surface_pressure",
    "precipitation", "wind_speed_10m", "wind_direction_10m",
    "boundary_layer_height"
]

def weather_archive(lat, lon, start_date, end_date):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": pd.Timestamp(start_date).date().isoformat(),
        "end_date": pd.Timestamp(end_date).date().isoformat(),
        "hourly": ",".join(WEATHER_VARS),
        "timezone": "UTC",
    }
    return http_get_json(OPEN_METEO_ARCHIVE, params=params)

def weather_forecast(lat, lon, forecast_days=2):
    params = {
        "latitude": lat, "longitude": lon,
        "hourly": ",".join(WEATHER_VARS),
        "forecast_days": forecast_days,
        "timezone": "UTC",
    }
    return http_get_json(OPEN_METEO_FORECAST, params=params)

def normalize_weather(payload):
    h = payload.get("hourly", {})
    df = pd.DataFrame(h)
    if "time" in df:
        df["timestamp"] = pd.to_datetime(df.pop("time"), utc=True)
    return df

In [ ]:
def grid_id(lat, lon, resolution_km=1.0):
    # Approximate 1-km grid using latitude-dependent longitude scaling.
    lat_step = resolution_km / 111.32
    lon_step = resolution_km / (111.32 * np.cos(np.deg2rad(np.clip(lat, -89, 89))))
    ilat = np.floor(lat / lat_step).astype(int)
    ilon = np.floor(lon / lon_step).astype(int)
    return pd.Series(ilat).astype(str) + "_" + pd.Series(ilon).astype(str)

def add_grid(df, lat_col="lat", lon_col="lon"):
    out = df.copy()
    out["grid_id"] = grid_id(out[lat_col].astype(float), out[lon_col].astype(float))
    return out

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dphi = np.deg2rad(lat2-lat1)
    dl = np.deg2rad(lon2-lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

## 1. Build a station time series

A full 1-km operational field should come from AeroPulse's materialized grid feature table. For a runnable public-data notebook, we use monitoring locations as the observed field and demonstrate wind-based transport between stations.

The forecast target is future PM2.5 at the destination station/grid.

In [ ]:
locations=discover_pm25_locations(max_locations=100)
END=pd.Timestamp.utcnow().floor("h")
START=END-pd.Timedelta(days=int(os.getenv("TRAIN_DAYS","60")))

records=[]
for _,loc in locations.head(int(os.getenv("MAX_STATIONS","30"))).iterrows():
    try:
        d=normalize_openaq_hours(openaq_location_hours(int(loc.location_id),START,END))
        d=d[d.parameter.isin(["pm25","pm2.5"])].copy()
        if not d.empty:
            d["lat"],d["lon"],d["station_name"]=loc.lat,loc.lon,loc["name"]
            records.append(d)
    except Exception as e: print("AQ failed",e)

aq=pd.concat(records,ignore_index=True)
aq["pm25"]=pd.to_numeric(aq.value,errors="coerce")
aq=aq.dropna(subset=["pm25"]).sort_values(["location_id","timestamp"])
print(aq.shape)

In [ ]:
# Add lags and future targets.
aq["pm25_lag1"]=aq.groupby("location_id").pm25.shift(1)
aq["pm25_lag3"]=aq.groupby("location_id").pm25.shift(3)
aq["pm25_lag6"]=aq.groupby("location_id").pm25.shift(6)
aq["pm25_roll6"]=aq.groupby("location_id").pm25.transform(lambda s:s.shift(1).rolling(6,min_periods=3).mean())

# 6-hour target; change HORIZON_HOURS to 1,3,6,12,24,48 and train separate models.
HORIZON_HOURS=int(os.getenv("HORIZON_HOURS","6"))
future=aq[["location_id","timestamp","pm25"]].copy()
future["target_time"]=future.timestamp-pd.Timedelta(hours=HORIZON_HOURS)
future=future.rename(columns={"pm25":"target_pm25"})
aq=aq.merge(future[["location_id","target_time","target_pm25"]],
            left_on=["location_id","timestamp"],right_on=["location_id","target_time"],how="left")

In [ ]:
weather_parts=[]
for (lat,lon),_ in aq.groupby(["lat","lon"]):
    try:
        weather_parts.append(normalize_weather(weather_archive(lat,lon,START.date(),END.date())).assign(lat=lat,lon=lon))
    except Exception as e: print("weather",e)
weather=pd.concat(weather_parts,ignore_index=True)
wd=np.deg2rad(weather.wind_direction_10m)
weather["wind_u"]=-np.sin(wd)*weather.wind_speed_10m
weather["wind_v"]=-np.cos(wd)*weather.wind_speed_10m
df=aq.merge(weather,on=["timestamp","lat","lon"],how="left")

# Approximate advection displacement over the horizon.
# Wind vector points toward movement direction after converting meteorological direction.
seconds=HORIZON_HOURS*3600
df["advect_km_u"]=df.wind_u*seconds/1000
df["advect_km_v"]=df.wind_v*seconds/1000
df["stability_proxy"]=df.boundary_layer_height/(df.wind_speed_10m.clip(lower=.5))

## 2. Baseline + ML residual correction

The deterministic baseline is persistence adjusted by wind/stability terms. The LightGBM model learns residual correction. This is more robust than asking an ML model to rediscover transport physics from scratch.

In [ ]:
FEATURES=[
    "pm25","pm25_lag1","pm25_lag3","pm25_lag6","pm25_roll6",
    "wind_u","wind_v","wind_speed_10m","wind_direction_10m",
    "boundary_layer_height","stability_proxy","temperature_2m",
    "relative_humidity_2m","precipitation","lat","lon"
]
FEATURES=[c for c in FEATURES if c in df.columns]
data=df.dropna(subset=["target_pm25"]).copy()

# Persistence baseline; residual model corrects the baseline.
data["baseline_forecast"]=data["pm25_lag1"].fillna(data["pm25"])
data["residual_target"]=data["target_pm25"]-data["baseline_forecast"]

cut=data.timestamp.quantile(.80)
train=data[data.timestamp<cut].copy()
test=data[data.timestamp>=cut].copy()

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from lightgbm import LGBMRegressor
import joblib

imp=SimpleImputer(strategy="median")
Xtr=imp.fit_transform(train[FEATURES])
Xte=imp.transform(test[FEATURES])
model=LGBMRegressor(
    n_estimators=700,learning_rate=.03,num_leaves=63,
    subsample=.85,colsample_bytree=.85,random_state=SEED
)
model.fit(Xtr,train.residual_target)

test["prediction"]=test.baseline_forecast+model.predict(Xte)
baseline_metrics={
    "MAE":mean_absolute_error(test.target_pm25,test.baseline_forecast),
    "RMSE":mean_squared_error(test.target_pm25,test.baseline_forecast,squared=False),
    "R2":r2_score(test.target_pm25,test.baseline_forecast)
}
model_metrics={
    "MAE":mean_absolute_error(test.target_pm25,test.prediction),
    "RMSE":mean_squared_error(test.target_pm25,test.prediction,squared=False),
    "R2":r2_score(test.target_pm25,test.prediction)
}
print("Baseline:",baseline_metrics)
print("Hybrid:",model_metrics)

## 3. Spatial/temporal validation

The LLD requires geography and season holdouts. The temporal split above is mandatory. Add station holdout validation before production; the cell below performs a station-group stress test when enough stations exist.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
tr_idx,te_idx=next(gss.split(data,groups=data.location_id))
gtrain=data.iloc[tr_idx]
gtest=data.iloc[te_idx]

gimp=SimpleImputer(strategy="median")
gXtr=gimp.fit_transform(gtrain[FEATURES])
gXte=gimp.transform(gtest[FEATURES])
gmodel=LGBMRegressor(n_estimators=500,learning_rate=.03,num_leaves=31,random_state=SEED)
gmodel.fit(gXtr,gtrain.residual_target)
gpred=gtest.baseline_forecast+gmodel.predict(gXte)

spatial_metrics={
    "MAE":mean_absolute_error(gtest.target_pm25,gpred),
    "RMSE":mean_squared_error(gtest.target_pm25,gpred,squared=False),
    "R2":r2_score(gtest.target_pm25,gpred)
}
spatial_metrics

In [ ]:
bundle={
    "model":model,"imputer":imp,"features":FEATURES,
    "horizon_hours":HORIZON_HOURS,
    "metrics":{"temporal_baseline":baseline_metrics,"temporal_hybrid":model_metrics,"spatial_holdout":spatial_metrics},
    "version":"1.0.0"
}
joblib.dump(bundle,ARTIFACT_DIR/f"propagation_forecast_{HORIZON_HOURS}h.joblib")
print("Saved forecast model.")

## 4. Live forecast

For live operation, the service should consume the current 1-km PM2.5 field, wind field, boundary-layer height, CAMS forecast and source-likelihood outputs from the materialized AeroPulse state. External APIs should be ingested asynchronously, not called from the user request path.

This cell demonstrates the same model contract against the latest available station observation.

In [ ]:
latest=[]
for _,loc in locations.head(min(10,len(locations))).iterrows():
    try:
        p=http_get_json(f"{OPENAQ_BASE}/locations/{int(loc.location_id)}/latest",
                        headers={"X-API-Key":require_env("OPENAQ_API_KEY")})
        for r in p.get("results",[]):
            if str((r.get("parameter") or {}).get("name","")).lower() in ("pm25","pm2.5"):
                dt=r.get("datetime") or r.get("period",{}).get("datetimeTo")
                dt=dt.get("utc") if isinstance(dt,dict) else dt
                latest.append({"timestamp":pd.to_datetime(dt,utc=True,errors="coerce"),
                               "location_id":loc.location_id,"lat":loc.lat,"lon":loc.lon,
                               "pm25":safe_float(r.get("value"))})
    except Exception as e: print(e)
live=pd.DataFrame(latest).dropna(subset=["timestamp","pm25"])
display(live.head())

In [ ]:
if not live.empty:
    r=live.sort_values("timestamp").iloc[-1]
    w=normalize_weather(weather_forecast(r.lat,r.lon,forecast_days=2))
    w=w.iloc[(w.timestamp-r.timestamp).abs().argmin()].to_dict()
    x={c:np.nan for c in bundle["features"]}
    x.update({
        "pm25":r.pm25,"pm25_lag1":r.pm25,"pm25_lag3":r.pm25,
        "pm25_lag6":r.pm25,"pm25_roll6":r.pm25,
        "wind_u":-np.sin(np.deg2rad(w.get("wind_direction_10m",np.nan)))*w.get("wind_speed_10m",np.nan),
        "wind_v":-np.cos(np.deg2rad(w.get("wind_direction_10m",np.nan)))*w.get("wind_speed_10m",np.nan),
        "wind_speed_10m":w.get("wind_speed_10m"),
        "wind_direction_10m":w.get("wind_direction_10m"),
        "boundary_layer_height":w.get("boundary_layer_height"),
        "temperature_2m":w.get("temperature_2m"),
        "relative_humidity_2m":w.get("relative_humidity_2m"),
        "precipitation":w.get("precipitation"),
        "lat":r.lat,"lon":r.lon
    })
    X=pd.DataFrame([x])[bundle["features"]]
    correction=bundle["model"].predict(bundle["imputer"].transform(X))[0]
    baseline=r.pm25
    forecast=max(0.0,baseline+correction)
    print({
        "origin_timestamp":str(r.timestamp),
        "horizon_hours":bundle["horizon_hours"],
        "baseline_pm25":float(baseline),
        "forecast_pm25":float(forecast),
        "ml_residual_correction":float(correction)
    })
else:
    print("No live observation available.")

## Production extension

Train separate horizon models for 1/3/6/12/24/48 hours, or move to a sequence model once the grid dataset is sufficiently dense. Compare every advanced model against this hybrid baseline. Release only when it demonstrates positive forecast skill across temporal, spatial and seasonal holdouts.